#  Pure data case

We have three hyperparameters for the model:

- Relationship between $\sigma_s$ and $\sigma_0$
  - These follow the relation $\sigma_x^2 = \sigma_0^2\sigma_s^2$
  - We can set one of them to $c\sigma_{\epsilon}$, where $c$ is some constant deciding how you want to tune reconstruction penalty v/s regularisation
  - Or we can make them equal, and set both as: $\sigma_0 = \sigma_s = \sigma_x$
  - This removes $\sigma_{\epsilon}$ from the equation, do note that $\sigma_{\epsilon}$ is itself set with respect to $\sigma_x$, which we assume has 1 std.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import shelve
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import *
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
)
from pt_to_api import disjoint_ae, disjoint_ae_learned_sig
from sklearn.metrics.pairwise import (
    pairwise_distances,
    cosine_similarity,
    cosine_distances,
)
from scipy.optimize import linear_sum_assignment
from collections import defaultdict
import numpy as np
from torch import nn
from torch import optim
import warnings
from dataclasses import dataclass
from typing import Any
import math
import gc
import pandas as pd
from pt_to_api import benchmark as B
import ast

MODE = "light"
SHELVE_CACHE_ROOT = Path.cwd() / "pure-case-v3"
SHELVE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

## Helpers for plotting

In [ ]:
import matplotlib.lines as mlines


def plot_reconstruction_quality(df, key):
    """
    df must contain: dims, atom_ratio, n_sample_ratio, term3, and the column specified by key
    """
    dims_vals  = sorted(df["dims"].unique())
    atom_vals  = sorted(df["atom_ratio"].unique())
    term3_vals = sorted(df["term3"].unique())

    colors  = ["#2196F3", "#FF9800", "#4CAF50", "#E91E63", "#9C27B0"]
    markers = ["o", "s", "^", "D", "v"]

    color_map  = {a: colors[i] for i, a in enumerate(atom_vals)}
    marker_map = {a: markers[i] for i, a in enumerate(atom_vals)}

    n_rows = len(dims_vals)
    n_cols = len(term3_vals)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(5 * n_cols, 4 * n_rows),
        sharex=False,
        sharey="row",
        constrained_layout=True
    )
    if n_cols == 1 and n_rows == 1:
        axes = np.array([axes])
    if n_cols == 1:
        axes = axes.reshape(-1, 1)
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    for row, dims in enumerate(dims_vals):
        for col, term3 in enumerate(term3_vals):
            sub = df[(df["dims"] == dims) & (df["term3"] == term3)]
            ax  = axes[row, col]

            for atom in atom_vals:
                d = sub[sub["atom_ratio"] == atom].sort_values("n_sample_ratio")
                ax.plot(
                    d["n_sample_ratio"], d[key],
                    marker=marker_map[atom], color=color_map[atom],
                    linewidth=2, markersize=7,
                    label=f"atom_ratio={atom}"
                )

            if row == 0:
                ax.set_title(f"term3 = {term3}", fontsize=12, fontweight="bold")
            if col == 0:
                ax.set_ylabel(f"dims={dims}\n{key}", fontsize=10)

            ax.set_xlabel("n_sample_ratio", fontsize=10)
            ax.grid(True, which="both", linestyle="--", alpha=0.4)
            ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
            ax.set_xticks(sorted(df["n_sample_ratio"].unique()))

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(atom_vals), fontsize=10,
               bbox_to_anchor=(0.5, -0.02), frameon=True)

    fig.suptitle(f"{key} vs. sample size\nby dims and term3", fontsize=14, fontweight="bold")

    plt.savefig(f"{key}_reconstruction_quality.png", dpi=150, bbox_inches="tight")
    plt.show()

def plot_mse_vs_meansim_multi(dfs, titles, suptitle, clip_quantile=None):
    n = len(dfs)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), constrained_layout=True)
    fig.suptitle(suptitle, fontweight="bold")

    if n == 1:
        axes = [axes]

    for ax, df, title in zip(axes, dfs, titles):
        ax.scatter(df["mse"], df["mean_sim"], s=60, alpha=0.7, color="#2196F3")
        ax.set_xlim(left=0, right=df["mse"].quantile(0.95))
        ax.set_xlabel("MSE ↓", fontsize=11)
        ax.set_ylabel("mean_sim ↑", fontsize=11)
        ax.set_title(title, fontsize=12)
        if clip_quantile:
            ax.set_xlim(left=0, right=df["mse"].quantile(clip_quantile))
        ax.grid(True, linestyle="--", alpha=0.4)

    plt.savefig("mse_vs_meansim_multi.png", dpi=150, bbox_inches="tight")
    plt.show()

# Start experiment

In [ ]:

def get_device(dim):
    if dim < 100:
        return "cpu"
    else:
        return "mps"



def get_metrics_df(metrics):
    mets = []
    for k, v in metrics.items():
        k = ast.literal_eval(k)
        m = v.copy()
        m["dims"] = k[0]
        m["atom_ratio"] = k[1]
        m["n_sample_ratio"] = k[2]
        m["term3"] = k[3]
        mets.append(m)
    
    df = pd.DataFrame(mets)
    df = df.drop(columns=["vec_sim"])
    
    return df

In [ ]:
def make_dim_partition(patch_dim, n_components, seed=42):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(patch_dim)
    return [list(perm[i::n_components]) for i in range(n_components)]

def generate_synthetic_patches(
    patch_dim=72,
    n_components=10,
    k=3,
    n_samples=1000,
    noise_std=0.01,
    seed=42,
    sigma_x=1,
):
    rng = np.random.RandomState(seed)

    dim_partition = make_dim_partition(patch_dim, n_components, seed)

    # ground truth atoms, nonzero only on owned dims
    W_true = np.zeros((n_components, patch_dim))
    for i, dims in enumerate(dim_partition):
        W_true[i, dims] = rng.randn(len(dims))
    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)

    # each sample uses at most k atoms
    codes_true = np.zeros((n_samples, n_components))
    for i in range(n_samples):
        k_i = rng.randint(1, k + 1)  # active atoms: 1..k
        idx = rng.choice(n_components, k_i, replace=False)
        codes_true[i, idx] = rng.randn(k_i)

    X = codes_true @ W_true

    scale = sigma_x / X.std()
    X *= scale
    W_true *= scale  # keeps codes_true @ W_true ≈ X
    X += rng.randn(*X.shape) * noise_std

    return X, W_true, codes_true, dim_partition

In [ ]:
def run_single_test(dims_list, atoms_ratio, samples_ratio, term3_set, kwarg_key, metrics):
    for dim in dims_list:
        for a in atoms_ratio:
            for sr in samples_ratio:
                for term3 in term3_set:                
                    
                    key = str((dim, a, sr, term3))
                    print("#######", key)
                    
                    atoms = math.ceil(dim*a)
                    k = atoms
                    n_samples = sr*atoms
                    if atoms == 1:
                        print("skip: atoms=1")
                        continue
                    if key in metrics:
                        print("skip: already done, delete the shelve file if you want to do it all over again")
                        continue
        
                    device = get_device(dim)
                    print("using", dim, atoms, k, n_samples, 0.01)
                    X, W_true, codes_true, dim_partition = generate_synthetic_patches(dim, atoms, k, n_samples=n_samples, noise_std=0.01)
            
                    scaler = B.MeanPerDimGlobalStdScaler().fit(X)
                    X_scaled = scaler.transform(X)
    
                    mets = []
                    for run_idx in range(NUM_RUNS_PER_TEST):
                        print("RUN:", run_idx)
                        kwargs = {kwarg_key: term3}
                        run = B.train(X_scaled, atoms, 1e-2, epochs=4000, device=device, **kwargs)
                        mets.append(B.get_metrics_from_run(run, W_true))
                    gc.collect()

                    
                    metrics[key] = B.aggregate_metrics(mets)

In [ ]:
NUM_RUNS_PER_TEST = 3

# original
DIMS_LIST = [10, 100]
ATOMS_RATIO = [0.1, 0.5, 0.9]    # for each dim, we test for 20%, 50% atoms for now, to understand how i can read the data
SAMPLES_RATIO = [1, 5, 20, 50]


# test, comment for actual work
# DIMS_LIST = [10,]
# ATOMS_RATIO = [0.5, 0.9]    # for each dim, we test for 20%, 50% atoms for now, to understand how i can read the data
# SAMPLES_RATIO = [20]

# Relation between $\sigma_s$ and $\sigma_0$

We can either set $\sigma_s$ to $c\sigma_{\epsilon}$ and derive $\sigma_0$ from it. or the other way round.  
This will determine which one is bigger.  

The relationship we use is simply $\sigma_0^2\sigma_s^2 = \sigma_x^2$ 

- We can either set one of them relative to $sigma_eps$ and derive the other
- We can make them equal, ie. $\sigma_0 = \sigma_s = \sqrt{\sigma_x}$

In [ ]:
import math

term3_set = ["equal", "less", "greater"]
kwarg_key = "sigma_s_rel_to_0"

# dims_list, atoms_ratio, samples_ratio, term3_set, kwarg_key, metrics

with shelve.open(SHELVE_CACHE_ROOT / "t_sigma_s_rel_to_0") as shelf:
    run_single_test(DIMS_LIST, ATOMS_RATIO, SAMPLES_RATIO, term3_set, kwarg_key, shelf)
    metrics = dict(shelf)


## Results

If you look at the table, you'll see that the sigma_s_to_0 gives mean simimilary (the column `mean_sim`) = 99%.  

Surprisingly, we see that if we set $\sigma_s < \sigma_0$, `dims=10` gives 55% mean similarity. The reason is the corresponding MSE. Its much higher than the equal case, we are not able to reconstruct well, so every atom is just a disjoint noise.  

For `dims=100`, $\sigma_s > \sigma_0$, we see 30-60% similarity. The MSE is fine too. On closer inspection, you'll see that the disjoint loss has driven all atoms to near 0. If you look at individual components of a single reconstruction, you'll see random atoms lighting up. This is because the codes have too much variance and are basically taking care of getting the reconstruction.  

It's hard to say why this happens. It's best to ignore that. For now, $\sigma_s = \sigma_0$ gives the best results.  
It also has the good property to simply remove the relation with $\sigma_eps$, which we can use to freely guide reconstruction.  

The next experiments will use the "equal" method

In [ ]:
df = get_metrics_df(metrics)

In [ ]:
# all equal values
df[df["term3"] == "equal"]

In [ ]:
# all less
df[df["term3"] == "less"]

In [ ]:
# all greater
df[df["term3"] == "greater"]

# log term usefulness

Closely inspecting the gradients generally tells us that the log term has very small gradients through out.  
The log term is basically pushing in the opposite direction from disjointness. It wants to keep weights "non-zero".  
The signal is very low though compared to the actual disjoint term. And we anyways rely on correct reconstruction to give us non-zero weights.  

This section checks if descent becomes easier in the synthetic case if we give up the log term. It is useful to only check the results in the previous section with the `sigma_s_to_0 = equal` case.  


In [ ]:
term3_set = [True, False]
kwarg_key = "use_ln_term"



with shelve.open(SHELVE_CACHE_ROOT / "t_use_ln") as shelf:
    run_single_test(DIMS_LIST, ATOMS_RATIO, SAMPLES_RATIO, term3_set, kwarg_key, shelf)
    metrics = dict(shelf)


In [ ]:
df = get_metrics_df(metrics)
df

## Results

We don't see a lot of difference. Although removing the `ln` component did bring down `mean_sim` in one case (row 3).  
It has also degraded the MSE in some cases, so this is not very conclusive.  

We will need to test this on other datasets to see if there is problem with the component. Otherwise we go ahead while staying faithful to the probabilistic model.  

# Initialisations

Three kinds of initialisations:

- Standard initialisation, starts decoder and encoder with from Gaussian, mean=0 and heuristics for sigma
- SVD: initialise decoder using SVD, with sigma using the same heuristics as standard. mean=0, encoder done in the same way as standard
- Warmup: initialise using standard, then do warmup epochs on L2 regularisation (without Weight loss). Then start training

In [ ]:
term3_set = [B.StandardInitStrategy(), B.SvdInitStrategy(), B.WarmupInitStrategy(warmup_epochs=3000)]
# only doing warmup, will merge dict later, without throwing away the optimizer
# term3_set = [B.WarmupInitStrategy(warmup_epochs=3000)]
kwarg_key = "init_strategy"

with shelve.open(SHELVE_CACHE_ROOT / "t_inits") as shelf:
    run_single_test(DIMS_LIST, ATOMS_RATIO, SAMPLES_RATIO, term3_set, kwarg_key, shelf)
    metrics = dict(shelf)


In [ ]:
df = get_metrics_df(metrics)
df

## Results

not much difference in the pure case

# Analysis


In [ ]:
# collect metrics

with shelve.open(SHELVE_CACHE_ROOT / "t_use_ln") as shelf:
    use_ln_df = get_metrics_df(dict(shelf))

with shelve.open(SHELVE_CACHE_ROOT / "t_inits") as shelf:
    inits_df = get_metrics_df(dict(shelf))

with shelve.open(SHELVE_CACHE_ROOT / "t_sigma_s_rel_to_0") as shelf:
    sigma_s20_df = get_metrics_df(dict(shelf))


## MSE v/s Mean similarity


We see a general tendency of Mean similarity to grow as we get MSE closer to 0.  
This makes sense, a good reconstruction has better chance to give us the correct atoms.  

Below, we have the plots of with MSE on the x-axis, and Mean similarity on the y axis.  

We have separate plots for every hyperparameter we used.  

This is also a very helpful metric. Hyperparameters which give good results on only good reconstructions lets us use reconstruction accuracy as a good indicator.  

In [ ]:
# d = sigma_s20_df
_g = lambda k: sigma_s20_df[sigma_s20_df["term3"] == k]
plot_mse_vs_meansim_multi([_g("equal"), _g("less"), _g("greater")], ["equal", "greater", "lesser"], "sigma_s_0")

_g = lambda k: use_ln_df[use_ln_df["term3"] == k]
plot_mse_vs_meansim_multi([_g(True), _g(False)], ["True", "False"], "use_ln")

_g = lambda k: inits_df[inits_df["term3"] == k]
plot_mse_vs_meansim_multi([_g("StandardInitStrategy"), _g("SvdInitStrategy"), _g("WarmupInitStrategy(3000)")], ["StandardInitStrategy", "SvdInitStrategy", "WarmupInitStrategy"], "init_strategies")

Below are the same plots, but clipped on the x-axis, this lets us see the values closer to 0.  
It is evident that jumping from `0.8` to `0.9+` requires very faithful reconstruction (the MSE has to be very close to the true noise, which was 0.01 in the generation process).  

In [ ]:
# d = sigma_s20_df
_g = lambda k: sigma_s20_df[sigma_s20_df["term3"] == k]
plot_mse_vs_meansim_multi([_g("equal"), _g("less"), _g("greater")], ["equal", "greater", "lesser"], "sigma_s_0", 0.5)

_g = lambda k: use_ln_df[use_ln_df["term3"] == k]
plot_mse_vs_meansim_multi([_g(True), _g(False)], ["True", "False"], "use_ln", 0.5)

_g = lambda k: inits_df[inits_df["term3"] == k]
plot_mse_vs_meansim_multi([_g("StandardInitStrategy"), _g("SvdInitStrategy"), _g("WarmupInitStrategy(3000)")], ["StandardInitStrategy", "SvdInitStrategy", "WarmupInitStrategy"], "init_strategies", 0.5)

## $\sigma_s$ v/s $\sigma_0$

The plot below shows mean similarity on y-axis.  Each row has the number of dimensions set (first row has dims=10).  
The x-axis contains `n-samples-ratio`. Essentially the number of samples.  

We see that $\sigma_s = \sigma_0$ (or `term3 = equal` in the graph) giving faithful results consistently.  

`dim=100` also shows us mean similarity increasing slowly as n_samples is increased. It seems in higher dimensions, we get worse similarity metrics.  
The plot below that shows the MSE on the y-axis.  It is evident that the MSE is quite low, leading to poor results.   

In [ ]:
plot_reconstruction_quality(sigma_s20_df, "mean_sim")

In [ ]:
d = sigma_s20_df[sigma_s20_df["dims"]==100]
plot_reconstruction_quality(d[d["term3"] == "equal"], "mse")

## log term

We don't see any similarity differences. 

In [ ]:
plot_reconstruction_quality(use_ln_df, "mean_sim")

## Initialisation strategies

Not many differences here too.  

In [ ]:
plot_reconstruction_quality(inits_df, "mean_sim")
plot_reconstruction_quality(inits_df, "mse")

# Getting better MSE

$\sigma_{\epsilon}$ is the hyperparameter which gives relative importance to MSE. The standard value we have used until now was $\sigma_x/100$ (corresponding to `eps_ratio=100`).  
Let's try to increase that.  

We'll do the case `atom_ratio=0.9, dims=100, n_samples_ratio=50` with different `eps_ratio` values.  

In [ ]:
term3_set = [100, 500]
kwarg_key = "eps_ratio"



with shelve.open(SHELVE_CACHE_ROOT / "eps_ratio1") as shelf:
    run_single_test([100], [0.9], [50], term3_set, kwarg_key, shelf)
    metrics = dict(shelf)


In [ ]:
df = get_metrics_df(metrics)
df

In [ ]:
# its instructive to look at the components to see what is going wrong
# i expect most of the components have been drowned to 0 values, and we have S doing the hard work now. 
# there is a good amount of deviation between encoder_sigma_ratio and decoder_sigma_ratio.  
# I'm not sure how to fix that.  
# this is a warning sign. i need to think how i can maintain these ratios. 
# but this is because of the data generation process itself.
# without knowing how data was generated, its hard to guess. hmmm.

In [ ]:
# idk if there are dead atoms
metrics['(100, 0.9, 50, 1000)']["vec_sim"]

In [ ]:
term3_set = [100, 500]
kwarg_key = "eps_ratio"



with shelve.open(SHELVE_CACHE_ROOT / "eps_ratio2") as shelf:
    run_single_test([100], [0.1], [20], term3_set, kwarg_key, shelf)
    metrics = dict(shelf)


In [ ]:
df = get_metrics_df(metrics)
df

In [ ]:
# dim = 100
# a = 0.9
# sr = 50


# # key = str((dim, a, sr, term3))
# # print("#######", key)

# atoms = math.ceil(dim*a)
# k = atoms
# n_samples = sr*atoms

# device = "mps"
# # print("using", dim, atoms, k, n_samples, 0.01)
# X, W_true, codes_true, dim_partition = generate_synthetic_patches(dim, atoms, k, n_samples=n_samples, noise_std=0.01)

# scaler = B.MeanPerDimGlobalStdScaler().fit(X)
# X_scaled = scaler.transform(X)

# mets = []

In [ ]:
# run = B.train(X_scaled, atoms, 1e-2, epochs=3_000, device=device)#, init_strategy=B.SvdInitStrategy())
# B.get_metrics_from_run(run, W_true)